In [41]:
from pathlib import Path

# --- 1. Патчим aiquant/strategy/trade_plan.py ---
tp_path = Path("aiquant/strategy/trade_plan.py")
tp_code = tp_path.read_text(encoding="utf-8")

# Синхронизируем build_from_market строго с Triple Barrier Labeling
old_market = """        if side == "LONG":
            atr_stop = entry - base_stop_dist
            stop_loss = (
                min(atr_stop, swing_low - atr * 0.4)
                if (swing_low is not None and swing_low < entry)
                else atr_stop
            )
            take_profit = entry + (entry - stop_loss) * target_ratio
        else:
            atr_stop = entry + base_stop_dist
            stop_loss = (
                max(atr_stop, swing_high + atr * 0.4)
                if (swing_high is not None and swing_high > entry)
                else atr_stop
            )
            take_profit = entry - (stop_loss - entry) * target_ratio"""

new_market = """        # Strictly aligned with compute_triple_barrier_labels
        if side == "LONG":
            stop_loss = entry - base_stop_dist
            take_profit = entry + base_stop_dist * target_ratio
        else:
            stop_loss = entry + base_stop_dist
            take_profit = entry - base_stop_dist * target_ratio"""

if old_market in tp_code:
    tp_code = tp_code.replace(old_market, new_market)
    tp_path.write_text(tp_code, encoding="utf-8")
    print("SUCCESS: trade_plan.py aligned with labeling geometry.")
else:
    print("WARNING: Old market pattern in trade_plan.py not found, check manually.")

# --- 2. Патчим aiquant/strategy/backtest_execution.py ---
be_path = Path("aiquant/strategy/backtest_execution.py")
be_code = be_path.read_text(encoding="utf-8")

# Убираем ловушку преждевременного трейлинга при 0.7R
old_trail = """            if side == "LONG":
                if (b_high - plan.entry) >= 0.7 * risk_dist:
                    active_sl = max(active_sl, plan.entry + (plan.entry * 0.0004))
                hit_sl, hit_tp = b_low <= active_sl, b_high >= plan.take_profit
            else:
                if (plan.entry - b_low) >= 0.7 * risk_dist:
                    active_sl = min(active_sl, plan.entry - (plan.entry * 0.0004))
                hit_sl, hit_tp = b_high >= active_sl, b_low <= plan.take_profit"""

new_trail = """            # Strictly hold until original SL or TP (matching Triple Barrier definition)
            if side == "LONG":
                hit_sl = b_low <= plan.stop_loss
                hit_tp = b_high >= plan.take_profit
            else:
                hit_sl = b_high >= plan.stop_loss
                hit_tp = b_low <= plan.take_profit"""

if old_trail in be_code:
    be_code = be_code.replace(old_trail, new_trail)
    be_path.write_text(be_code, encoding="utf-8")
    print("SUCCESS: backtest_execution.py noise-trailing removed.")
else:
    print("WARNING: Trailing pattern in backtest_execution.py not found, check manually.")

SUCCESS: trade_plan.py aligned with labeling geometry.
SUCCESS: backtest_execution.py noise-trailing removed.
